In [ ]:
import pandas as pd
import math
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score, make_scorer, classification_report, confusion_matrix, roc_auc_score
from imblearn.combine import SMOTEENN
import joblib


def train_human_judge_model(train_file, model_out="best_lgb.pkl", thresh_out="best_thresh.npy"):
    df = pd.read_excel(train_file)

    features = ['BLEU', 'ROUGE-L', 'BERTScore', 'AI judge']
    #features = ['BLEU', 'ROUGE-L', 'BERTScore']
    epsilon = 1e-8
    for f in features:
        df[f] = df[f].apply(lambda x: math.log(x + epsilon))

    X = df[features]
    y = df['Human_Judge']

    # Train/val split
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, stratify=y, test_size=0.3, random_state=42
    )

    # Handle imbalance
    sm3 = SMOTEENN(random_state=42)
    X_train, y_train = sm3.fit_resample(X_train, y_train)

    # Grid Search
    param_grid = {
        'max_depth': [1, 2, 3],
        'learning_rate': [0.1, 0.2],
        'n_estimators': [50, 100],
        'num_leaves': [20, 50],
        'subsample': [0.5, 0.8]
    }
    lgb_clf = lgb.LGBMClassifier(objective='binary', random_state=42)
    f1_scorer = make_scorer(f1_score, pos_label=0)

    grid_search = GridSearchCV(
        estimator=lgb_clf,
        param_grid=param_grid,
        scoring=f1_scorer,
        cv=5,
        n_jobs=-1,
        verbose=0
    )
    grid_search.fit(X_train, y_train)

    best_lgb = grid_search.best_estimator_
    probs = best_lgb.predict_proba(X_val)[:, 1]

    # Threshold tuning
    thresholds = np.linspace(0, 1, 200)
    best_thresh, best_f1 = 0.5, 0.0
    for t in thresholds:
        preds = (probs >= t).astype(int)
        f1 = f1_score(y_val, preds, pos_label=0)
        if f1 > best_f1:
            best_f1, best_thresh = f1, t

    # Save model + threshold
    joblib.dump(best_lgb, model_out)
    np.save(thresh_out, best_thresh)

    print("Training done.")
    print("Best params:", grid_search.best_params_)
    print("Validation best F1:", best_f1)
    print("Best threshold:", best_thresh)

    return best_lgb, best_thresh


In [ ]:
def apply_human_judge_model(
    predict_file,
    model_in="best_lgb.pkl",
    thresh_in="best_thresh.npy",
    output_file="predictions.xlsx"
):
    import joblib, numpy as np, pandas as pd, math

    # Load model + threshold
    best_lgb = joblib.load(model_in)
    best_thresh = np.load(thresh_in)

    # Load new data
    df_new = pd.read_excel(predict_file)

    # Features
    features = ['BLEU', 'ROUGE-L', 'BERTScore', 'AI judge']
    #features = ['BLEU', 'ROUGE-L', 'BERTScore']
    epsilon = 1e-8
    for f in features:
        df_new[f] = df_new[f].apply(lambda x: math.log(x + epsilon))

    # Predict probabilities
    X_new = df_new[features]
    probs = best_lgb.predict_proba(X_new)[:, 1]

    # Apply threshold for binary labels
    preds = (probs >= best_thresh).astype(int)

    # Add new columns
    df_new["Predicted_Prob"] = probs
    df_new["Predicted_Human_Judge"] = preds

    # Save to Excel
    df_new.to_excel(output_file, index=False)
    print(f"Predictions saved to {output_file}")

    return df_new


In [ ]:
# Train once (with labels)
train_human_judge_model("Question_Pool_IAQ.xlsx")



[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 366, number of negative: 415
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000056 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 768
[LightGBM] [Info] Number of data points in the train set: 781, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.468630 -> initscore=-0.125645
[LightGBM] [Info] Start training from score -0.125645
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

(LGBMClassifier(learning_rate=0.2, max_depth=3, num_leaves=20,
                objective='binary', random_state=42, subsample=0.5),
 np.float64(0.11557788944723618))

In [ ]:
results_df = apply_human_judge_model("sampling_all_IAQ_together.xlsx")

Predictions saved to predictions.xlsx


In [ ]:
def report_classification_metrics(df, label_col="Human judge", pred_col="Predicted_Human_Judge"):
    """
    Report precision, recall, f1-score, and support for each class
    """
    from sklearn.metrics import classification_report, confusion_matrix

    if label_col not in df.columns:
        raise KeyError(f"Ground truth column '{label_col}' not found in DataFrame.")
    if pred_col not in df.columns:
        raise KeyError(f"Prediction column '{pred_col}' not found in DataFrame.")

    y_true = df[label_col]
    y_pred = df[pred_col]

    # Print confusion matrix
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

    # Print precision, recall, f1-score
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, digits=3))


In [ ]:
import pandas as pd

df = pd.read_excel("/content/Four_GPT_NSD.xlsx")
results_df = report_classification_metrics(df)


Confusion Matrix:
[[ 29  41]
 [ 51 749]]

Classification Report:
              precision    recall  f1-score   support

           0      0.362     0.414     0.387        70
           1      0.948     0.936     0.942       800

    accuracy                          0.894       870
   macro avg      0.655     0.675     0.664       870
weighted avg      0.901     0.894     0.897       870

